In [1]:
import os
import sys
import json
import random
from pathlib import Path
from typing import Any

import numpy as np
import torch
from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

from transformers import AutoImageProcessor, AutoProcessor, AutoTokenizer

try:
    from transformers import Qwen3VLForConditionalGeneration
except Exception:
    Qwen3VLForConditionalGeneration = None

try:
    from transformers import AutoModelForImageTextToText
except Exception:
    from transformers import AutoModelForVision2Seq as AutoModelForImageTextToText

PROJECT_ROOT = Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from src.emotion_reasoning.config import load_experiment_config
from src.emotion_reasoning.datasets import build_dataset
from src.emotion_reasoning.evaluation.ablation import run_ablation_suite
from src.emotion_reasoning.evaluation.attention_viz import aggregate_cross_attention, overlay_attention_on_image
from src.emotion_reasoning.modeling.multimodal_model import MultimodalEmotionModel
from src.emotion_reasoning.training.trainer import evaluate_model, train_experiment
from src.emotion_reasoning.utils.image_ops import draw_red_box, load_rgb_image
from src.emotion_reasoning.utils.io import load_checkpoint, load_records, save_json, save_records

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATASET_ROOT = PROJECT_ROOT / "caer_dataset" / "CAER-S"
WORKDIR = PROJECT_ROOT / "notebook_outputs" / "risetv1_qwen"
WORKDIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATASET_ROOT exists:", DATASET_ROOT.exists())
print("WORKDIR:", WORKDIR)
print("DEVICE:", DEVICE)

/home/agung/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PROJECT_ROOT: /home/agung/riset
DATASET_ROOT exists: True
WORKDIR: /home/agung/riset/notebook_outputs/risetv1_qwen
DEVICE: cuda:1


In [2]:
ALLOWED_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}
VAL_RATIO = 0.2

ANNOTATION_PATH = WORKDIR / "annotations" / "caers_annotations_with_val.jsonl"


def _stable_sample_id(rel_path: str, split_name: str) -> str:
    return f"{split_name}__{rel_path.replace('/', '__').replace('.', '_')}"


def _collect_split(split_name: str) -> list[dict[str, Any]]:
    split_dir = DATASET_ROOT / split_name
    if not split_dir.exists():
        return []

    rows: list[dict[str, Any]] = []
    for cls_dir in sorted(split_dir.iterdir()):
        if not cls_dir.is_dir():
            continue

        label_name = cls_dir.name
        for img_path in sorted(cls_dir.rglob("*")):
            if not img_path.is_file() or img_path.suffix.lower() not in ALLOWED_EXTS:
                continue

            rel_path = img_path.relative_to(DATASET_ROOT).as_posix()
            rows.append(
                {
                    "sample_id": _stable_sample_id(rel_path, split_name),
                    "image_path": rel_path,
                    "labels": label_name,
                    "split": split_name,
                    "bbox": None,
                }
            )
    return rows


train_raw = _collect_split("train")
test_raw = _collect_split("test")

if len(train_raw) == 0 or len(test_raw) == 0:
    raise FileNotFoundError(
        "Dataset CAER-S tidak ditemukan lengkap di caer_dataset/CAER-S/{train,test}."
    )

indices = np.arange(len(train_raw))
strat_labels = [row["labels"] for row in train_raw]
train_idx, val_idx = train_test_split(
    indices,
    test_size=VAL_RATIO,
    random_state=SEED,
    stratify=strat_labels,
)

train_records: list[dict[str, Any]] = []
val_records: list[dict[str, Any]] = []
for i in train_idx:
    row = dict(train_raw[int(i)])
    row["split"] = "train"
    row["sample_id"] = _stable_sample_id(row["image_path"], "train")
    train_records.append(row)

for i in val_idx:
    row = dict(train_raw[int(i)])
    row["split"] = "val"
    row["sample_id"] = _stable_sample_id(row["image_path"], "val")
    val_records.append(row)

test_records: list[dict[str, Any]] = []
for row in test_raw:
    copied = dict(row)
    copied["split"] = "test"
    copied["sample_id"] = _stable_sample_id(copied["image_path"], "test")
    test_records.append(copied)

all_records = train_records + val_records + test_records
save_records(ANNOTATION_PATH, all_records)

CLASS_NAMES = sorted({row["labels"] for row in all_records})
print("Annotation saved:", ANNOTATION_PATH)
print("Class names:", CLASS_NAMES)
print(
    "Split sizes:",
    {
        "train": len(train_records),
        "val": len(val_records),
        "test": len(test_records),
        "total": len(all_records),
    },
)

Annotation saved: /home/agung/riset/notebook_outputs/risetv1_qwen/annotations/caers_annotations_with_val.jsonl
Class names: ['Angry', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']
Split sizes: {'train': 39205, 'val': 9802, 'test': 20992, 'total': 69999}


In [3]:
# Reuse existing pseudo labels and normalize legacy "Anger" -> "Angry" entries
REUSE_STAGE1_PATH = PROJECT_ROOT / "notebook_outputs" / "stage1_pseudo_labels.jsonl"
STAGE1_OUTPUT_PATH = WORKDIR / "stage1_pseudo_labels_qwen_fixed.jsonl"

if not REUSE_STAGE1_PATH.exists():
    raise FileNotFoundError(f"Pseudo-label source tidak ditemukan: {REUSE_STAGE1_PATH}")

raw_stage1 = load_records(REUSE_STAGE1_PATH)
normalized_stage1: list[dict[str, Any]] = []
num_label_fixed = 0
num_path_fixed = 0

for row in raw_stage1:
    updated = dict(row)

    label_text = str(updated.get("labels", "")).strip()
    if label_text == "Anger":
        updated["labels"] = "Angry"
        num_label_fixed += 1

    image_ref = str(updated.get("image_path", ""))
    if "/Anger/" in image_ref:
        updated["image_path"] = image_ref.replace("/Anger/", "/Angry/")
        num_path_fixed += 1

    normalized_stage1.append(updated)

missing_paths = 0
for row in normalized_stage1:
    candidate = DATASET_ROOT / Path(str(row["image_path"]))
    if not candidate.exists():
        missing_paths += 1

if missing_paths > 0:
    raise FileNotFoundError(
        f"Masih ada {missing_paths} image path yang tidak valid setelah normalisasi stage-1."
    )

save_records(STAGE1_OUTPUT_PATH, normalized_stage1)

stage1_splits = {}
for row in normalized_stage1:
    split_name = str(row.get("split", "unknown"))
    stage1_splits[split_name] = stage1_splits.get(split_name, 0) + 1

CLASS_NAMES = sorted({str(row["labels"]) for row in normalized_stage1})
print("Reused Stage-1 source:", REUSE_STAGE1_PATH)
print("Normalized Stage-1 output:", STAGE1_OUTPUT_PATH)
print("Rows:", len(normalized_stage1))
print("Fixed labels (Anger->Angry):", num_label_fixed)
print("Fixed image paths (/Anger/->/Angry/):", num_path_fixed)
print("Split sizes:", stage1_splits)
print("Class names:", CLASS_NAMES)

Reused Stage-1 source: /home/agung/riset/notebook_outputs/stage1_pseudo_labels.jsonl
Normalized Stage-1 output: /home/agung/riset/notebook_outputs/risetv1_qwen/stage1_pseudo_labels_qwen_fixed.jsonl
Rows: 69999
Fixed labels (Anger->Angry): 2999
Fixed image paths (/Anger/->/Angry/): 2999
Split sizes: {'train': 44107, 'val': 4900, 'test': 20992}
Class names: ['Angry', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']


In [6]:
QWEN_MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
STAGE1_OUTPUT_PATH = WORKDIR / "stage1_pseudo_labels_qwen.jsonl"

RUN_STAGE1 = True
MAX_NEW_TOKENS = 96
SAVE_EVERY = 50
STAGE1_SAMPLE_LIMIT = None
RESUME_STAGE1 = True
USE_4BIT = False
cache_dir = PROJECT_ROOT / "model_cache"

PROMPT_WITH_BOX_TEMPLATE = (
    "Given the following list of emotions: {emotion_list}, please explain in detail "
    "which emotions are most suitable for describing how the person in the red box feels "
    "based on image context. Focus on facial expressions, body pose, and environmental cues."
)

PROMPT_NO_BOX_TEMPLATE = (
    "Given the following list of emotions: {emotion_list}, please explain in detail "
    "which emotions are most suitable for describing how the main person feels based on "
    "image context. Focus on facial expressions, body pose, and environmental cues."
)


def _load_qwen_model(model_id: str) -> torch.nn.Module:
    dtype = torch.float16 if DEVICE.type == "cuda" else torch.float32
    model_kwargs: dict[str, Any] = {
        "torch_dtype": dtype,
        "low_cpu_mem_usage": True,
    }

    if USE_4BIT and DEVICE.type == "cuda":
        from transformers import BitsAndBytesConfig

        model_kwargs = {
            "quantization_config": BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=torch.float16,
            ),
            "device_map": "auto",
            "low_cpu_mem_usage": True,
        }

    if Qwen3VLForConditionalGeneration is not None:
        model = Qwen3VLForConditionalGeneration.from_pretrained(model_id, **model_kwargs, cache_dir=cache_dir)
    else:
        model = AutoModelForImageTextToText.from_pretrained(model_id, **model_kwargs)

    if not (USE_4BIT and DEVICE.type == "cuda"):
        model = model.to(DEVICE)

    model.eval()
    return model


def _generate_caption(
    model: torch.nn.Module,
    processor: AutoProcessor,
    image: Image.Image,
    prompt_text: str,
    max_new_tokens: int,
) -> str:
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt_text},
            ],
        }
    ]

    if hasattr(processor, "apply_chat_template"):
        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
        )
    else:
        fallback_prompt = f"USER: <image>\n{prompt_text}\nASSISTANT:"
        inputs = processor(images=image, text=fallback_prompt, return_tensors="pt")

    inputs = {
        key: value.to(model.device) if torch.is_tensor(value) else value
        for key, value in inputs.items()
    }

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    if "input_ids" in inputs:
        output_ids = output_ids[:, inputs["input_ids"].shape[1] :]

    return processor.batch_decode(
        output_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()


emotion_list_text = ", ".join(CLASS_NAMES)
prompt_with_box = PROMPT_WITH_BOX_TEMPLATE.format(emotion_list=emotion_list_text)
prompt_no_box = PROMPT_NO_BOX_TEMPLATE.format(emotion_list=emotion_list_text)

records = load_records(ANNOTATION_PATH)
if STAGE1_SAMPLE_LIMIT is not None:
    records = records[: int(STAGE1_SAMPLE_LIMIT)]

if RUN_STAGE1:
    existing_by_id: dict[str, dict[str, Any]] = {}
    if RESUME_STAGE1 and STAGE1_OUTPUT_PATH.exists():
        for row in load_records(STAGE1_OUTPUT_PATH):
            sid = str(row.get("sample_id", ""))
            if sid:
                existing_by_id[sid] = row

    print("Preparing Qwen processor/model...")
    processor = AutoProcessor.from_pretrained(QWEN_MODEL_ID, trust_remote_code=True)
    model = _load_qwen_model(QWEN_MODEL_ID)
    print("Stage-1 device:", model.device)

    results_by_id: dict[str, dict[str, Any]] = dict(existing_by_id)
    errors: list[tuple[str, str]] = []

    for idx, row in enumerate(tqdm(records, desc="Stage-1 pseudo-label generation")):
        sample_id = str(row.get("sample_id", f"sample_{idx:06d}"))
        if sample_id in results_by_id:
            continue

        try:
            image_ref = Path(str(row["image_path"]))
            image_path = image_ref if image_ref.is_absolute() else DATASET_ROOT / image_ref
            image = load_rgb_image(image_path)

            bbox = row.get("bbox")
            if bbox not in (None, "", "nan"):
                image_for_prompt = draw_red_box(image, bbox)
                prompt_text = prompt_with_box
            else:
                image_for_prompt = image
                prompt_text = prompt_no_box

            generated_text = _generate_caption(
                model=model,
                processor=processor,
                image=image_for_prompt,
                prompt_text=prompt_text,
                max_new_tokens=MAX_NEW_TOKENS,
            )

            updated = dict(row)
            updated["semantic_pseudo_label"] = generated_text
            results_by_id[sample_id] = updated

        except Exception as exc:
            failed = dict(row)
            failed["semantic_pseudo_label"] = ""
            failed["stage1_error"] = str(exc)
            results_by_id[sample_id] = failed
            errors.append((sample_id, str(exc)))

        if (idx + 1) % SAVE_EVERY == 0:
            ordered_partial = []
            for r_idx, original in enumerate(records):
                sid = str(original.get("sample_id", f"sample_{r_idx:06d}"))
                if sid in results_by_id:
                    ordered_partial.append(results_by_id[sid])
            save_records(STAGE1_OUTPUT_PATH, ordered_partial)

    ordered_results = []
    for r_idx, original in enumerate(records):
        sid = str(original.get("sample_id", f"sample_{r_idx:06d}"))
        if sid in results_by_id:
            ordered_results.append(results_by_id[sid])

    save_records(STAGE1_OUTPUT_PATH, ordered_results)
    print("Stage-1 output:", STAGE1_OUTPUT_PATH)
    print("Generated records:", len(ordered_results))
    print("Generation errors:", len(errors))
    if errors:
        print("First 3 errors:", errors[:3])
else:
    print("RUN_STAGE1=False. Expecting existing file at:", STAGE1_OUTPUT_PATH)
    if not STAGE1_OUTPUT_PATH.exists():
        raise FileNotFoundError(
            "Stage-1 output tidak ditemukan. Set RUN_STAGE1=True atau siapkan file output terlebih dulu."
        )

Preparing Qwen processor/model...


KeyboardInterrupt: 

In [4]:
CONFIG_PATH = WORKDIR / "configs" / "caers_qwen_qformer.json"
OUTPUT_DIR = WORKDIR / "stage2_qformer"

config_payload = {
    "experiment_name": "caers_qwen_qformer_notebook",
    "seed": SEED,
    "dataset": {
        "name": "caer-s",
        "annotation_path": str(STAGE1_OUTPUT_PATH),
        "image_root": str(DATASET_ROOT),
        "task_type": "singlelabel",
        "image_column": "image_path",
        "label_column": "labels",
        "split_column": "split",
        "sample_id_column": "sample_id",
        "bbox_column": "bbox",
        "pseudo_label_column": "semantic_pseudo_label",
        "train_split": "train",
        "val_split": "val",
        "test_split": "test",
        "num_workers": 2,
        "max_text_length": 96,
        "class_names": CLASS_NAMES,
    },
    "model": {
        "vision_encoder_name": "openai/clip-vit-base-patch32",
        "text_encoder_name": "roberta-base",
        "num_queries": 32,
        "qformer_hidden_size": 512,
        "qformer_num_layers": 4,
        "qformer_num_heads": 8,
        "dropout": 0.3,
        "fusion_mode": "multimodal",
        "freeze_vision_encoder": False,
        "freeze_text_encoder": False,
    },
    "training": {
        "batch_size": 8,
        "epochs": 12,
        "gradient_clip_norm": 1.0,
        "mixed_precision": True,
        "early_stopping_patience": 4,
        "weight_decay": 0.05,
        "vision_lr": 1e-5,
        "text_lr": 1e-4,
        "qformer_lr": 1e-4,
        "head_lr": 1e-3,
        "output_dir": str(OUTPUT_DIR),
    },
}

save_json(CONFIG_PATH, config_payload)
experiment_config = load_experiment_config(CONFIG_PATH)

print("Config saved:", CONFIG_PATH)
print(json.dumps(config_payload, indent=2))

Config saved: /home/agung/riset/notebook_outputs/risetv1_qwen/configs/caers_qwen_qformer.json
{
  "experiment_name": "caers_qwen_qformer_notebook",
  "seed": 42,
  "dataset": {
    "name": "caer-s",
    "annotation_path": "/home/agung/riset/notebook_outputs/risetv1_qwen/stage1_pseudo_labels_qwen_fixed.jsonl",
    "image_root": "/home/agung/riset/caer_dataset/CAER-S",
    "task_type": "singlelabel",
    "image_column": "image_path",
    "label_column": "labels",
    "split_column": "split",
    "sample_id_column": "sample_id",
    "bbox_column": "bbox",
    "pseudo_label_column": "semantic_pseudo_label",
    "train_split": "train",
    "val_split": "val",
    "test_split": "test",
    "num_workers": 2,
    "max_text_length": 96,
    "class_names": [
      "Angry",
      "Disgust",
      "Fear",
      "Happy",
      "Neutral",
      "Sad",
      "Surprise"
    ]
  },
  "model": {
    "vision_encoder_name": "openai/clip-vit-base-patch32",
    "text_encoder_name": "roberta-base",
    "num_

In [ ]:
TRAIN_STAGE2 = True

if not STAGE1_OUTPUT_PATH.exists():
    raise FileNotFoundError(
        f"Stage-1 output belum ada: {STAGE1_OUTPUT_PATH}. Jalankan Cell 3 dulu."
    )

if TRAIN_STAGE2:
    train_results = train_experiment(experiment_config)
    print("Training finished.")
    print(json.dumps(train_results, indent=2))
else:
    train_results = None
    print("TRAIN_STAGE2=False, skip training.")

BEST_CHECKPOINT = Path(experiment_config.training.output_dir) / "best.pt"
RESULTS_JSON = Path(experiment_config.training.output_dir) / "results.json"
HISTORY_JSON = Path(experiment_config.training.output_dir) / "history.json"

print("Best checkpoint:", BEST_CHECKPOINT, "| exists:", BEST_CHECKPOINT.exists())
print("Results file:", RESULTS_JSON, "| exists:", RESULTS_JSON.exists())
print("History file:", HISTORY_JSON, "| exists:", HISTORY_JSON.exists())

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 62969.94it/s]
CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEX

In [ ]:
RUN_EVAL = True
RUN_ABLATION = False

if not BEST_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Best checkpoint belum ditemukan: {BEST_CHECKPOINT}. Jalankan training dulu."
    )

if RUN_EVAL:
    eval_config = load_experiment_config(CONFIG_PATH)
    test_metrics = evaluate_model(
        config=eval_config,
        checkpoint_path=BEST_CHECKPOINT,
        split="test",
        fusion_mode="multimodal",
    )
    print("Test metrics (multimodal):")
    print(json.dumps(test_metrics, indent=2))
else:
    test_metrics = None
    print("RUN_EVAL=False, skip evaluation.")

if RUN_ABLATION:
    ablation_config = load_experiment_config(CONFIG_PATH)
    ablation_summary = run_ablation_suite(ablation_config)
    print("Ablation summary:")
    print(json.dumps(ablation_summary, indent=2))
else:
    ablation_summary = None
    print("RUN_ABLATION=False, skip ablation (set True to run vision/text/multimodal ablation).")

ablation_file = (
    Path(experiment_config.training.output_dir).parent
    / f"{Path(experiment_config.training.output_dir).name}_ablation_summary.json"
)
print("Ablation file:", ablation_file, "| exists:", ablation_file.exists())

In [ ]:
ATTENTION_OUTPUT = WORKDIR / "stage2_qformer" / "attention_overlay_test0.png"
GENERATE_ATTENTION = True

if GENERATE_ATTENTION:
    viz_config = load_experiment_config(CONFIG_PATH)
    test_dataset = build_dataset(viz_config.dataset, split="test")
    if len(test_dataset) == 0:
        raise ValueError("Test dataset kosong, tidak bisa membuat attention overlay.")

    sample = test_dataset[0]
    tokenizer = AutoTokenizer.from_pretrained(viz_config.model.text_encoder_name, use_fast=True)
    image_processor = AutoImageProcessor.from_pretrained(viz_config.model.vision_encoder_name)

    text_inputs = tokenizer(
        [sample["text"] or ""],
        padding=True,
        truncation=True,
        max_length=viz_config.dataset.max_text_length,
        return_tensors="pt",
    )
    image_inputs = image_processor(images=[sample["image"]], return_tensors="pt")

    model_viz = MultimodalEmotionModel(viz_config.model, num_classes=viz_config.num_classes).to(DEVICE)
    checkpoint = load_checkpoint(BEST_CHECKPOINT, map_location=DEVICE)
    model_viz.load_state_dict(checkpoint["model_state_dict"])
    model_viz.eval()

    with torch.inference_mode():
        outputs = model_viz(
            pixel_values=image_inputs["pixel_values"].to(DEVICE),
            input_ids=text_inputs["input_ids"].to(DEVICE),
            attention_mask=text_inputs["attention_mask"].to(DEVICE),
            output_attentions=True,
        )

    attention_grid = aggregate_cross_attention(outputs["cross_attentions"])[0]
    saved_attention = overlay_attention_on_image(
        image=sample["image"],
        attention_grid=attention_grid,
        output_path=ATTENTION_OUTPUT,
        alpha=0.45,
    )
    print("Attention overlay saved:", saved_attention)
else:
    saved_attention = None
    print("GENERATE_ATTENTION=False, skip attention visualization.")

alignment_check = {
    "stage0_annotation_has_train_val_test": ANNOTATION_PATH.exists(),
    "stage1_vlm_pseudo_label_generated": STAGE1_OUTPUT_PATH.exists(),
    "stage1_output_column_semantic_pseudo_label": True,
    "stage1_prompt_uses_emotion_list": True,
    "stage1_red_box_if_bbox_available": True,
    "stage2_qformer_from_repo": True,
    "stage2_differential_lr_optimizer": True,
    "stage2_metrics_map_auc_accuracy": True,
    "ablation_available": True,
    "cross_attention_visualization_available": saved_attention is not None,
}

print("\nAlignment checklist:")
for key, value in alignment_check.items():
    print(f"- {key}: {value}")

print("\nKey artifacts:")
print("- Annotation:", ANNOTATION_PATH)
print("- Stage1 output:", STAGE1_OUTPUT_PATH)
print("- Config:", CONFIG_PATH)
print("- Best checkpoint:", BEST_CHECKPOINT)
print("- Results:", RESULTS_JSON)
print("- Attention:", ATTENTION_OUTPUT)